# Chart → data test (Branch A) — extract-line-chart-data on Colab

Goal: test whether `extract-line-chart-data` (LineFormer + ChartDete) can pull data
points from our catalysis **performance line plots**, as a step toward structure–activity records.

**Before running:** Runtime → Change runtime type → **GPU (T4)**. Free tier is enough.

**Input images:** single-panel line charts (PNG/JPEG). Composite multi-panel figures
(e.g. a 4-panel (a)(b)(c)(d) figure) must be **cropped into single panels first** —
LineFormer expects one chart per image.

**Honest caveat:** the install uses MMDetection (`mmcv-full` + an `mmdet` fork). On a
current Colab CUDA/torch this is the step most likely to need version fiddling. If the
install cell fails, see the Troubleshooting section at the bottom.

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
# 2. Clone the repo
%cd /content
!rm -rf extract-line-chart-data
!git clone https://github.com/tdsone/extract-line-chart-data.git
%cd extract-line-chart-data
!ls

In [ ]:
# 3. Install. The repo targets Python 3.10 + uv; on Colab we use pip directly.
#    setup_local_env.sh runs `mim install mmcv-full`, clones ChartDete into
#    third_party/, and installs its mmdet fork. This is the fragile step.
!pip install -q openmim
!pip install -q -e ".[local]"
!bash setup_local_env.sh
print('\n--- install finished (check above for errors) ---')

### If the install above errored on mmcv / mmdet
Pin versions to Colab's torch. Run this cell, then re-run the install cell:
```python
import torch
cu = 'cu' + torch.version.cuda.replace('.', '')   # e.g. cu121
tv = torch.__version__.split('+')[0]              # e.g. 2.3.0
print(f'pin mmcv for torch{tv}/{cu}')
!pip install -q mmengine
!pip install -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/{cu}/torch{tv}/index.html
```
MMDetection forks can lag current torch; if it stays broken, fall back to the
Claude-Vision path (see Troubleshooting at the very bottom).

In [ ]:
# 4. Download the model checkpoints the repo needs (LineFormer + ChartDete).
#    Follow the repo README; many versions fetch these in setup_local_env.sh.
#    If checkpoints are NOT auto-downloaded, the README lists Google Drive links
#    — download them into the paths the repo expects, then continue.
!find . -name '*.pth' -maxdepth 4 2>/dev/null | head
print('If no .pth files listed, fetch checkpoints per the repo README before step 6.')

In [ ]:
# 5. Upload your cropped single-panel chart PNGs into ./input
import os
os.makedirs('input', exist_ok=True)
from google.colab import files
up = files.upload()                 # pick one or more PNG/JPEG files
for name in up:
    os.replace(name, os.path.join('input', name))
print('input/ now contains:', os.listdir('input'))

In [ ]:
# 6. Run the pipeline
from plextract import extract
extract(input_dir='input', output_dir='output', backend='local')
print('\ndone — see output/ below')
!find output -name 'data.json' | head

In [ ]:
# 7. Inspect the extracted data + overlay
import json, glob
from IPython.display import Image, display
for dj in glob.glob('output/**/converted_datapoints/data.json', recursive=True):
    print('=== ', dj, ' ===')
    print(json.dumps(json.load(open(dj)), indent=2)[:1500])
# show any overlay/debug image the pipeline produced
for img in glob.glob('output/**/lineformer/*.png', recursive=True)[:4]:
    display(Image(img))

## What to judge
1. **Did it find the right number of curves?** (one per catalyst series)
2. **Are the (x, y) points close to the real values?** Compare against a manual
   WebPlotDigitizer trace of the same plot (the ground-truth check).
3. **Did it read the axis labels / legend?** (`axis_label_texts.json`). The hard part
   for us is mapping each curve → catalyst name; note how well that works.

## Troubleshooting
- **mmcv / mmdet install fails:** run the version-pin cell, re-run install. MMDetection
  forks often lag the latest torch — if unfixable, this tool may not run on current Colab.
- **Composite figures:** crop to single panels first (LineFormer = one chart per image).
- **Fallback (zero GPU/setup):** a Claude-Vision reader can run on the Mac via the
  Anthropic API we already use — lower coordinate precision, but it reads legends/labels
  well and needs no MMDetection. Ask Claude to scaffold `scripts/analysis/vision_chart.py`.